# Understat -> Nation Scrape -> Merge

This notebook runs the full pipeline:
1. Pull Understat Big-5 player-season data and save base files.
2. Scrape player nation data with free sources and fallbacks:
   - Wikidata citizenship (`P27`)
   - Wikidata description nationality parsing
   - Wikipedia extract nationality parsing
3. Merge `player_country_map.csv` into the full and WC-filtered datasets.
4. Export unresolved WC-filtered players.

Run cells top-to-bottom.

In [1]:
from pathlib import Path
import pandas as pd

from understat import (
    init_understat,
    load_understat_player_season_stats,
    standardize_understat_player_stats,
    filter_wc_players,
    MIN_MINUTES_DEFAULT,
)

# --- Paths ---
BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / "wc2026_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FULL_CSV = OUT_DIR / "understat_big5_player_season_2122_2526_full.csv"
WC_CSV = OUT_DIR / "understat_big5_player_season_2122_2526_wc_filtered.csv"
COUNTRY_MAP_CSV = BASE_DIR / "player_country_map.csv"
FULL_MERGED_CSV = OUT_DIR / "understat_big5_player_season_2122_2526_full_merged_nations.csv"
WC_MERGED_CSV = OUT_DIR / "understat_big5_player_season_2122_2526_wc_filtered_merged_nations.csv"
MISSING_WC_CSV = OUT_DIR / "wc_filtered_players_missing_nation.csv"

# --- API / scrape settings ---
WIKIDATA_API = "https://www.wikidata.org/w/api.php"
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"
WIKIPEDIA_API = "https://en.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "wc2026-research/1.0 (contact: somakshivlani@gmail.com)"}

REQUEST_TIMEOUT_S = 30
WIKIDATA_SEARCH_LIMIT = 8
WIKIPEDIA_SEARCH_LIMIT = 5
SCRAPE_SLEEP_S = 0.25
RESCRAPE_MISSING_DEFAULT = True

# --- Demonym mapping ---
DEMONYM_TO_COUNTRY = {
    "algerian": "Algeria", "argentine": "Argentina", "argentinian": "Argentina",
    "australian": "Australia", "austrian": "Austria", "belgian": "Belgium",
    "bolivian": "Bolivia", "brazilian": "Brazil", "cameroonian": "Cameroon",
    "canadian": "Canada", "chilean": "Chile", "colombian": "Colombia",
    "croatian": "Croatia", "czech": "Czech Republic", "danish": "Kingdom of Denmark",
    "dutch": "Netherlands", "ecuadorean": "Ecuador", "ecuadorian": "Ecuador",
    "egyptian": "Egypt", "english": "England", "estonian": "Estonia",
    "finnish": "Finland", "french": "France", "gambian": "The Gambia",
    "german": "Germany", "ghanaian": "Ghana", "greek": "Greece",
    "guinean": "Guinea", "hungarian": "Hungary", "icelandic": "Iceland",
    "iranian": "Iran", "iraqi": "Iraq", "irish": "Ireland",
    "israeli": "Israel", "italian": "Italy", "ivorian": "Ivory Coast",
    "japanese": "Japan", "korean": "South Korea", "kosovan": "Kosovo",
    "mexican": "Mexico", "moroccan": "Morocco", "nigerian": "Nigeria",
    "norwegian": "Norway", "paraguayan": "Paraguay", "peruvian": "Peru",
    "polish": "Poland", "portuguese": "Portugal", "romanian": "Romania",
    "russian": "Russia", "scottish": "Scotland", "senegalese": "Senegal",
    "serbian": "Serbia", "slovak": "Slovakia", "slovenian": "Slovenia",
    "south african": "South Africa", "spanish": "Spain", "swedish": "Sweden",
    "swiss": "Switzerland", "tunisian": "Tunisia", "turkish": "Turkey",
    "ukrainian": "Ukraine", "uruguayan": "Uruguay", "uzbek": "Uzbekistan",
    "welsh": "Wales",
}

print("Base dir:", BASE_DIR)
print("Output dir:", OUT_DIR)
print("Config loaded: paths + API constants + demonym map")

[04/07/26 14:27:00] INFO     No custom team name replacements found. You can configure these in       ]8;id=881986;file:///Users/somakshivlani/Documents/CSC642/Project/.venv/lib/python3.13/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=583935;file:///Users/somakshivlani/Documents/CSC642/Project/.venv/lib/python3.13/site-packages/soccerdata/_config.py#92\92]8;;\
                             /Users/somakshivlani/soccerdata/config/teamname_replacements.json.                    

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=554863;file:///Users/somakshivlani/Documents/CSC642/Project/.venv/lib/python3.13/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=955032;file:///Users/somakshivlani/Documents/CSC642/Project/.venv/lib/python3.13/site-packages/soccerdata/_config.py#198\198]8;;\
                             /Users/somakshivlani/soccerdata/config/league_dict.json.                              

Base dir: /Users/somakshivlani/Documents/CSC642/Project
Output dir: /Users/somakshivlani/Documents/CSC642/Project/wc2026_outputs
Config loaded: paths + API constants + demonym map


In [ ]:
# 1) Understat scrape + save full and WC-filtered CSVs
us = init_understat()
raw_df = load_understat_player_season_stats(us)
full_df = standardize_understat_player_stats(raw_df)
wc_df = filter_wc_players(full_df, min_minutes=MIN_MINUTES_DEFAULT)

full_df.to_csv(FULL_CSV, index=False)
wc_df.to_csv(WC_CSV, index=False)

print(f"Saved full: {FULL_CSV} ({len(full_df):,} rows)")
print(f"Saved wc filtered: {WC_CSV} ({len(wc_df):,} rows)")

In [8]:
# 2) Nation scrape (free-only fallback chain) and update/create player_country_map.csv
import re
import time
import requests
import unicodedata

# Uses constants from cell 1: API endpoints, headers, limits, and DEMONYM_TO_COUNTRY.


def strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def normalize_text(text: str) -> str:
    txt = strip_accents(text).lower()
    txt = re.sub(r"[^a-z\s-]", " ", txt)
    return re.sub(r"\s+", " ", txt).strip()


def parse_nation_from_text(text: str):
    if not text:
        return None
    normalized = normalize_text(text)

    # Prioritize UK federations before generic "british"
    for key in ["english", "welsh", "scottish", "irish"]:
        if re.search(rf"\b{re.escape(key)}\b", normalized):
            return DEMONYM_TO_COUNTRY[key]

    # Generic demonym fallback
    for demonym, country in DEMONYM_TO_COUNTRY.items():
        if re.search(rf"\b{re.escape(demonym)}\b", normalized):
            return country

    # If text says british and no region found, keep UK-level label
    if re.search(r"\bbritish\b", normalized):
        return "United Kingdom"

    return None


def wikidata_search_player(name: str, limit: int = WIKIDATA_SEARCH_LIMIT):
    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": "en",
        "uselang": "en",
        "type": "item",
        "limit": limit,
        "search": name,
    }
    r = requests.get(WIKIDATA_API, params=params, headers=HEADERS, timeout=REQUEST_TIMEOUT_S)
    r.raise_for_status()
    return r.json().get("search", [])


def get_country_from_qid(qid: str):
    query = f"""
    SELECT ?countryLabel WHERE {{
      wd:{qid} wdt:P27 ?country .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT 5
    """
    r = requests.get(
        WIKIDATA_SPARQL,
        params={"query": query, "format": "json"},
        headers=HEADERS,
        timeout=REQUEST_TIMEOUT_S,
    )
    r.raise_for_status()
    bindings = r.json()["results"]["bindings"]
    if not bindings:
        return None
    return bindings[0]["countryLabel"]["value"]


def choose_best_candidate(name: str, candidates: list[dict]):
    if not candidates:
        return None

    football_words = ["footballer", "soccer player", "association football", "football player"]
    score_rows = []
    normalized_name = normalize_text(name)

    for c in candidates:
        label = c.get("label") or ""
        desc = c.get("description") or ""
        score = 0

        if any(word in desc.lower() for word in football_words):
            score += 10
        if normalize_text(label) == normalized_name:
            score += 3
        if any(bad in desc.lower() for bad in ["rugby", "cricket", "politician", "wrestler"]):
            score -= 6

        score_rows.append((score, c))

    score_rows.sort(key=lambda x: x[0], reverse=True)
    return score_rows[0][1]


def wikipedia_extract_text(title: str):
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "exintro": 1,
        "explaintext": 1,
        "titles": title,
    }
    r = requests.get(WIKIPEDIA_API, params=params, headers=HEADERS, timeout=REQUEST_TIMEOUT_S)
    r.raise_for_status()
    pages = r.json().get("query", {}).get("pages", {})
    for page in pages.values():
        extract = page.get("extract")
        if extract:
            return extract
    return None


def wikipedia_search_player(name: str):
    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srlimit": WIKIPEDIA_SEARCH_LIMIT,
        "srsearch": f"{name} footballer",
    }
    r = requests.get(WIKIPEDIA_API, params=params, headers=HEADERS, timeout=REQUEST_TIMEOUT_S)
    r.raise_for_status()
    return r.json().get("query", {}).get("search", [])


def get_wikipedia_nation(name: str):
    results = wikipedia_search_player(name)
    for hit in results:
        title = hit.get("title")
        if not title:
            continue
        text = wikipedia_extract_text(title)
        nation = parse_nation_from_text(text or "")
        if nation:
            return nation, title
    return None, None


def resolve_player_country(name: str, sleep_s: float = SCRAPE_SLEEP_S):
    candidates = wikidata_search_player(name)
    best = choose_best_candidate(name, candidates)

    if not best:
        wiki_nation, wiki_title = get_wikipedia_nation(name)
        time.sleep(sleep_s)
        return {
            "player": name,
            "wikidata_id": None,
            "wikidata_label": None,
            "wikidata_description": None,
            "nation": wiki_nation,
            "nation_source": f"wikipedia:{wiki_title}" if wiki_nation else None,
        }

    qid = best.get("id")
    description = best.get("description")

    # Fallback order: P27 -> Wikidata description demonym -> Wikipedia extract demonym
    nation = get_country_from_qid(qid) if qid else None
    source = "wikidata_p27" if nation else None

    if not nation:
        nation = parse_nation_from_text(description or "")
        if nation:
            source = "wikidata_description"

    if not nation:
        wiki_nation, wiki_title = get_wikipedia_nation(name)
        nation = wiki_nation
        if nation:
            source = f"wikipedia:{wiki_title}"

    time.sleep(sleep_s)

    return {
        "player": name,
        "wikidata_id": qid,
        "wikidata_label": best.get("label"),
        "wikidata_description": description,
        "nation": nation,
        "nation_source": source,
    }


def build_or_update_country_map(df: pd.DataFrame, player_col: str = "player", rescrape_missing: bool = RESCRAPE_MISSING_DEFAULT) -> pd.DataFrame:
    players = sorted(df[player_col].dropna().astype(str).str.strip().unique())

    if COUNTRY_MAP_CSV.exists():
        cache = pd.read_csv(COUNTRY_MAP_CSV)
    else:
        cache = pd.DataFrame(columns=[
            "player", "wikidata_id", "wikidata_label", "wikidata_description", "nation", "nation_source"
        ])

    for col in ["player", "wikidata_id", "wikidata_label", "wikidata_description", "nation", "nation_source"]:
        if col not in cache.columns:
            cache[col] = None

    cache["player"] = cache["player"].astype(str).str.strip()
    latest = cache.drop_duplicates(subset=["player"], keep="last")

    def is_blank(value):
        return pd.isna(value) or str(value).strip() == ""

    if rescrape_missing:
        todo_players = [
            p for p in players
            if p not in set(latest["player"]) or is_blank(latest.loc[latest["player"] == p, "nation"].iloc[0])
        ]
    else:
        done_players = set(latest["player"]) if not latest.empty else set()
        todo_players = [p for p in players if p not in done_players]

    rows = []
    for i, player in enumerate(todo_players, start=1):
        print(f"[{i}/{len(todo_players)}] Resolving {player}")
        try:
            rows.append(resolve_player_country(player))
        except Exception as e:
            rows.append(
                {
                    "player": player,
                    "wikidata_id": None,
                    "wikidata_label": None,
                    "wikidata_description": f"ERROR: {e}",
                    "nation": None,
                    "nation_source": "error",
                }
            )

    if rows:
        fresh = pd.DataFrame(rows)
        cache = cache[~cache["player"].isin(fresh["player"])].copy()
        cache = pd.concat([cache, fresh], ignore_index=True)
        cache.to_csv(COUNTRY_MAP_CSV, index=False)

    return cache


'''country_map_df = build_or_update_country_map(
    wc_df,
    player_col="player",
    rescrape_missing=RESCRAPE_MISSING_DEFAULT,
)
print("Country map saved to:", COUNTRY_MAP_CSV)
print("Rows in country map:", len(country_map_df))
print("Rows with nation:", int(country_map_df["nation"].notna().sum()))'''

'country_map_df = build_or_update_country_map(\n    wc_df,\n    player_col="player",\n    rescrape_missing=RESCRAPE_MISSING_DEFAULT,\n)\nprint("Country map saved to:", COUNTRY_MAP_CSV)\nprint("Rows in country map:", len(country_map_df))\nprint("Rows with nation:", int(country_map_df["nation"].notna().sum()))'

In [9]:
# 3) Merge nation map into full and wc-filtered files + export unresolved list
import csv

# Uses constants from cell 1: FULL_MERGED_CSV, WC_MERGED_CSV, MISSING_WC_CSV.


def merge_nations(in_csv: Path, out_csv: Path, map_csv: Path):
    name_to_nation = {}
    with map_csv.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            player = (row.get("player") or "").strip()
            nation = (row.get("nation") or "").strip()
            if player and nation:
                name_to_nation[player] = nation

    rows = []
    matched = 0
    unmatched = 0

    with in_csv.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames

        for row in reader:
            player = (row.get("player") or "").strip()
            mapped_nation = name_to_nation.get(player, "")

            if mapped_nation:
                row["nation"] = mapped_nation
                matched += 1
            else:
                row["nation"] = ""
                unmatched += 1

            rows.append(row)

    with out_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"Merged -> {out_csv}")
    print(f"total_rows={len(rows)}, matched={matched}, still_blank={unmatched}")
    return rows


def export_missing_wc_players(rows: list[dict], out_csv: Path):
    missing_rows = []
    for row in rows:
        nation = (row.get("nation") or "").strip()
        if nation == "":
            missing_rows.append({"player": (row.get("player") or "").strip()})

    with out_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["player"])
        writer.writeheader()
        writer.writerows(missing_rows)

    print(f"Missing-player CSV -> {out_csv}")
    print(f"missing_rows={len(missing_rows)}, unique_players={len({r['player'] for r in missing_rows if r['player']})}")


merge_nations(FULL_CSV, FULL_MERGED_CSV, COUNTRY_MAP_CSV)
wc_rows = merge_nations(WC_CSV, WC_MERGED_CSV, COUNTRY_MAP_CSV)
export_missing_wc_players(wc_rows, MISSING_WC_CSV)

Merged -> /Users/somakshivlani/Documents/CSC642/Project/wc2026_outputs/understat_big5_player_season_2122_2526_full_merged_nations.csv
total_rows=13837, matched=9885, still_blank=3952
Merged -> /Users/somakshivlani/Documents/CSC642/Project/wc2026_outputs/understat_big5_player_season_2122_2526_wc_filtered_merged_nations.csv
total_rows=7719, matched=7236, still_blank=483
Missing-player CSV -> /Users/somakshivlani/Documents/CSC642/Project/wc2026_outputs/wc_filtered_players_missing_nation.csv
missing_rows=483, unique_players=222


In [10]:
# 4) Re-scrape only unresolved WC players, then re-merge + re-export missing list

# Requires cells 1, 3, and 4 to be run first (functions/constants loaded).

wc_merged_df = pd.read_csv(WC_MERGED_CSV)
missing_players = sorted(
    wc_merged_df.loc[
        wc_merged_df["nation"].isna() | (wc_merged_df["nation"].astype(str).str.strip() == ""),
        "player",
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print(f"Missing players to re-scrape: {len(missing_players)}")

if missing_players:
    retry_df = pd.DataFrame({"player": missing_players})
    country_map_df = build_or_update_country_map(retry_df, player_col="player", rescrape_missing=True)
    print("Country map updated:", COUNTRY_MAP_CSV)
    print("Rows in country map:", len(country_map_df))

    # Re-run merge and unresolved export
    merge_nations(FULL_CSV, FULL_MERGED_CSV, COUNTRY_MAP_CSV)
    wc_rows = merge_nations(WC_CSV, WC_MERGED_CSV, COUNTRY_MAP_CSV)
    export_missing_wc_players(wc_rows, MISSING_WC_CSV)
else:
    print("No missing players found in WC merged file.")

Missing players to re-scrape: 222
[1/222] Resolving Aarón
[2/222] Resolving Abduqodir Khusanov
[3/222] Resolving Abner
[4/222] Resolving Adrian Beck
[5/222] Resolving Alaa Bellarouch
[6/222] Resolving Aleksandar Pavlovic
[7/222] Resolving Aleksander Sedlar
[8/222] Resolving Alfon
[9/222] Resolving Alfonso
[10/222] Resolving Alpha Touré
[11/222] Resolving Alvaro Rodríguez
[12/222] Resolving Amadou Koné
[13/222] Resolving André
[14/222] Resolving Anton Kade
[15/222] Resolving Antonio Martínez
[16/222] Resolving Antony
[17/222] Resolving Arne Engels
[18/222] Resolving Arthur
[19/222] Resolving Augusto Batalla
[20/222] Resolving Azz-Eddine Ounahi
[21/222] Resolving Balenziaga
[22/222] Resolving Ben Godfrey
[23/222] Resolving Ben Osborn
[24/222] Resolving Ben Seghir
[25/222] Resolving Ben White
[26/222] Resolving Bence Dárdai
[27/222] Resolving Benedict Hollerbach
[28/222] Resolving Benjamin André
[29/222] Resolving Benno Schmitz
[30/222] Resolving Benoit Badiashile Mukinayi
[31/222] Resolv

In [2]:
# 5) Build season-player-player_id master CSV for API merges
# One row per (season, player), choosing the row with highest minutes
# when duplicates exist (e.g., transfers).

SEASON_PLAYER_ID_CSV = OUT_DIR / "understat_big5_player_season_player_player_id_master.csv"

src = pd.read_csv(FULL_MERGED_CSV)

required_cols = ["season", "player", "player_id"]
missing = [c for c in required_cols if c not in src.columns]
if missing:
    raise ValueError(f"Missing required columns in source CSV: {missing}")

minutes_col = "minutes" if "minutes" in src.columns else None

work = src.copy()
work["player"] = work["player"].astype(str).str.strip()

if minutes_col:
    work[minutes_col] = pd.to_numeric(work[minutes_col], errors="coerce").fillna(0)
    work = work.sort_values(["season", "player", minutes_col], ascending=[True, True, False])
else:
    work = work.sort_values(["season", "player"]) 

season_player_id = (
    work.drop_duplicates(subset=["season", "player"], keep="first")
        [["season", "player", "player_id"]]
        .sort_values(["season", "player"])
        .reset_index(drop=True)
)

season_player_id.to_csv(SEASON_PLAYER_ID_CSV, index=False)

print(f"Saved: {SEASON_PLAYER_ID_CSV}")
print(f"Rows: {len(season_player_id):,}")
print("Unique seasons:", season_player_id["season"].nunique())
print("Unique players:", season_player_id["player"].nunique())

Saved: /Users/somakshivlani/Documents/CSC642/Project/wc2026_outputs/understat_big5_player_season_player_player_id_master.csv
Rows: 13,450
Unique seasons: 5
Unique players: 5580


In [4]:
# 6) Pull StatsBomb open-data international tournament stats and merge onto Understat seasons
#
# Season mapping (new season starts August):
#   WC 2022      Nov–Dec 2022  → season 2223
#   AFCON 2023   Jan–Feb 2023  → season 2223
#   Euro 2024    Jun–Jul 2024  → season 2324  (user rule: summer tourn. belongs to preceding season)
#   Copa America 2024  Jun–Jul 2024  → season 2324
#
# Output: wc2026_outputs/statsbomb_intl_player_season_totals.csv
#         Columns: player, season, sb_competition, sb_minutes, sb_goals, sb_assists,
#                  sb_shots, sb_xg, sb_xa, sb_np_shots, sb_np_xg, sb_key_passes,
#                  sb_progressive_carries, sb_progressive_passes

from statsbombpy import sb
import warnings
warnings.filterwarnings("ignore")

SB_OUT = OUT_DIR / "statsbomb_intl_player_season_totals.csv"

# Competitions to pull: (competition_id, season_id, label, understat_season_code)
INTL_COMPS = [
    (43, 106, "FIFA World Cup 2022",    "2223"),
    (1267, 107, "AFCON 2023",           "2223"),
    (55, 282, "UEFA Euro 2024",         "2324"),
    (223, 282, "Copa America 2024",     "2324"),
]

def _agg_statsbomb_events(competition_id: int, season_id: int) -> pd.DataFrame:
    """Aggregate StatsBomb event data to player-level totals for one tournament."""
    try:
        matches_df = sb.matches(competition_id=competition_id, season_id=season_id, fmt="dataframe")
    except Exception as e:
        print(f"  Could not load matches: {e}")
        return pd.DataFrame()

    rows = []
    for match_id in matches_df["match_id"]:
        try:
            events = sb.events(match_id=match_id, fmt="dataframe", flatten_attrs=False)
        except Exception:
            continue

        # --- minutes played (approximate from last event timestamp per player/team combo) ---
        lineups_raw = sb.lineups(match_id=match_id)
        for team_name, lineup_df in lineups_raw.items():
            for _, player_row in lineup_df.iterrows():
                player_name = player_row.get("player_name") or player_row.get("player", {})
                if isinstance(player_name, dict):
                    player_name = player_name.get("name", "")

                player_events = events[events["player"].apply(
                    lambda x: (x.get("name") if isinstance(x, dict) else x) == player_name
                    if pd.notna(x) else False
                )]

                if player_events.empty:
                    continue

                minutes = float(player_events["minute"].max() or 0)

                shots = player_events[player_events["type"].apply(
                    lambda x: (x.get("name") if isinstance(x, dict) else x) == "Shot"
                    if pd.notna(x) else False
                )]

                def _outcome(s):
                    o = s.get("shot", {}) if isinstance(s, dict) else {}
                    if isinstance(o, dict):
                        return o.get("outcome", {}).get("name", "") if isinstance(o.get("outcome"), dict) else str(o.get("outcome", ""))
                    return ""

                goals = sum(1 for _, s in shots.iterrows() if _outcome(s.to_dict()) == "Goal")
                xg_total = sum(
                    (s.get("shot", {}) or {}).get("statsbomb_xg", 0) or 0
                    if isinstance(s, dict) else 0
                    for _, s in shots.iterrows()
                )
                np_shots_mask = shots.apply(
                    lambda r: (r.get("shot", {}) or {}).get("type", {}).get("name", "") not in ("Penalty",)
                    if isinstance(r.get("shot"), dict) else True, axis=1
                )
                np_shots = int(np_shots_mask.sum())
                np_xg = sum(
                    (s.get("shot", {}) or {}).get("statsbomb_xg", 0) or 0
                    if isinstance(s, dict) and (s.get("shot", {}) or {}).get("type", {}).get("name", "") != "Penalty"
                    else 0
                    for _, s in shots.iterrows()
                )

                passes = player_events[player_events["type"].apply(
                    lambda x: (x.get("name") if isinstance(x, dict) else x) == "Pass"
                    if pd.notna(x) else False
                )]
                assists = sum(
                    1 for _, p in passes.iterrows()
                    if isinstance(p.get("pass"), dict)
                    and (p["pass"].get("goal_assist") is True or p["pass"].get("assisted_shot_id") is not None
                         and p["pass"].get("goal_assist") is True)
                )
                xa_total = sum(
                    (p.get("pass", {}) or {}).get("expected_assists_added", 0) or 0
                    if isinstance(p.get("pass"), dict) else 0
                    for _, p in passes.iterrows()
                )
                key_passes = sum(
                    1 for _, p in passes.iterrows()
                    if isinstance(p.get("pass"), dict) and p["pass"].get("shot_assist") is True
                )
                prog_passes = sum(
                    1 for _, p in passes.iterrows()
                    if isinstance(p.get("pass"), dict)
                    and (p["pass"].get("length", 0) or 0) >= 10
                    and not (p["pass"].get("switch") or p["pass"].get("cross"))
                )

                carries = player_events[player_events["type"].apply(
                    lambda x: (x.get("name") if isinstance(x, dict) else x) == "Carry"
                    if pd.notna(x) else False
                )]
                prog_carries = sum(
                    1 for _, c in carries.iterrows()
                    if isinstance(c.get("carry"), dict)
                    and (c["carry"].get("end_location", [0, 0])[0] or 0) > (c.get("location", [0, 0]) or [0, 0])[0]
                )

                rows.append({
                    "player":              player_name,
                    "sb_nation":           team_name,
                    "sb_minutes":          minutes,
                    "sb_goals":            goals,
                    "sb_xg":               round(xg_total, 4),
                    "sb_np_xg":            round(np_xg, 4),
                    "sb_shots":            len(shots),
                    "sb_np_shots":         np_shots,
                    "sb_assists":          assists,
                    "sb_xa":               round(xa_total, 4),
                    "sb_key_passes":       key_passes,
                    "sb_progressive_passes":  prog_passes,
                    "sb_progressive_carries": prog_carries,
                })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    # sb_nation: keep the one team name per player (should always be the same)
    agg = {c: "sum" for c in df.columns if c not in ("player", "sb_nation")}
    agg["sb_nation"] = "first"
    return df.groupby("player", as_index=False).agg(agg)


all_frames = []
for comp_id, season_id, label, us_season in INTL_COMPS:
    print(f"\n{'='*55}")
    print(f"Pulling: {label}  (season code → {us_season})")
    print(f"{'='*55}")
    df = _agg_statsbomb_events(comp_id, season_id)
    if df.empty:
        print(f"  No data returned for {label}.")
        continue
    df["sb_competition"] = label
    df["season"] = us_season
    all_frames.append(df)
    print(f"  Players aggregated: {len(df):,}")

if all_frames:
    intl_df = pd.concat(all_frames, ignore_index=True)
    intl_df.to_csv(SB_OUT, index=False)
    print(f"\nSaved: {SB_OUT}  ({len(intl_df):,} rows)")
    print(intl_df[["player", "season", "sb_competition", "sb_minutes", "sb_goals", "sb_xg"]].head(10).to_string(index=False))
else:
    print("No StatsBomb data collected. Check your statsbombpy install: pip install statsbombpy")


Pulling: FIFA World Cup 2022  (season code → 2223)
  Players aggregated: 679

Pulling: AFCON 2023  (season code → 2223)
  Players aggregated: 511

Pulling: UEFA Euro 2024  (season code → 2324)
  Players aggregated: 496

Pulling: Copa America 2024  (season code → 2324)
  Players aggregated: 339

Saved: /Users/somakshivlani/Documents/CSC642/Project/wc2026_outputs/statsbomb_intl_player_season_totals.csv  (2,025 rows)
                           player season      sb_competition  sb_minutes  sb_goals  sb_xg
                       Aaron Mooy   2223 FIFA World Cup 2022       380.0         0      0
                     Aaron Ramsey   2223 FIFA World Cup 2022       278.0         0      0
                Abdelhamid Sabiri   2223 FIFA World Cup 2022       419.0         1      0
Abdelkarim Hassan Al Haj Fadlalla   2223 FIFA World Cup 2022       277.0         0      0
             Abderrazak Hamdallah   2223 FIFA World Cup 2022       375.0         0      0
            Abdessamad Ezzalzouli   2223 

In [5]:
# 7) Merge StatsBomb international totals into the Understat full-merged dataset
#
# Join key: season + player (fuzzy-tolerant: strip whitespace, case-fold)
# One row per (player, season, league/team) in Understat is enriched with
# sb_* columns. Players with no international data get NaN sb_* columns.

import unicodedata

MERGED_WITH_SB_CSV = OUT_DIR / "understat_big5_player_season_2122_2526_full_merged_intl.csv"

def _norm(s: str) -> str:
    """Normalize player name for matching: lowercase, strip accents, trim spaces."""
    s = str(s).strip().lower()
    s = "".join(
        c for c in unicodedata.normalize("NFKD", s)
        if not unicodedata.combining(c)
    )
    return " ".join(s.split())

intl_df = pd.read_csv(SB_OUT)
understat_df = pd.read_csv(FULL_MERGED_CSV)

# Build normalized key columns for joining
intl_df["_key"] = intl_df["player"].map(_norm)
intl_df["_season"] = intl_df["season"].astype(str)

understat_df["_key"] = understat_df["player"].map(_norm)
understat_df["_season"] = understat_df["season"].astype(str)

# Aggregate: if a player appeared in >1 tournament in the same season, sum sb_* columns
sb_cols = [c for c in intl_df.columns if c.startswith("sb_") and c not in ("sb_competition", "sb_nation")]
sb_competitions_col = (
    intl_df.groupby(["_key", "_season"])["sb_competition"]
    .apply(lambda x: " | ".join(sorted(set(x))))
    .reset_index()
    .rename(columns={"sb_competition": "sb_competitions"})
)
sb_nation_col = (
    intl_df.groupby(["_key", "_season"])["sb_nation"]
    .first()
    .reset_index()
)
intl_agg = (
    intl_df.groupby(["_key", "_season"], as_index=False)[sb_cols].sum()
    .merge(sb_competitions_col, on=["_key", "_season"], how="left")
    .merge(sb_nation_col, on=["_key", "_season"], how="left")
)

merged = understat_df.merge(
    intl_agg,
    on=["_key", "_season"],
    how="left",
)

# Drop temp join keys
merged = merged.drop(columns=["_key", "_season"])

merged.to_csv(MERGED_WITH_SB_CSV, index=False)

matched = merged[merged["sb_minutes"].notna()]
print(f"Saved: {MERGED_WITH_SB_CSV}")
print(f"Total rows:   {len(merged):,}")
print(f"Rows with intl data: {len(matched):,}  ({matched['player'].nunique():,} unique players)")
print(f"\nSample matched players:")
print(
    matched[["player", "season", "nation", "sb_competitions", "sb_minutes", "sb_goals", "sb_xg"]]
    .drop_duplicates(subset=["player", "season"])
    .head(12)
    .to_string(index=False)
)

Saved: /Users/somakshivlani/Documents/CSC642/Project/wc2026_outputs/understat_big5_player_season_2122_2526_full_merged_intl.csv
Total rows:   13,837
Rows with intl data: 564  (449 unique players)

Sample matched players:
            player  season             nation                  sb_competitions  sb_minutes  sb_goals  sb_xg
       Bukayo Saka    2223            England              FIFA World Cup 2022       289.0       3.0    0.0
      Granit Xhaka    2223        Switzerland              FIFA World Cup 2022       379.0       0.0    0.0
  Leandro Trossard    2223            Belgium              FIFA World Cup 2022       240.0       0.0    0.0
 Takehiro Tomiyasu    2223              Japan              FIFA World Cup 2022       312.0       0.0    0.0
    William Saliba    2223             France              FIFA World Cup 2022       102.0       0.0    0.0
      Jan Bednarek    2223             Poland              FIFA World Cup 2022        94.0       0.0    0.0
Leander Dendoncker    2

In [ ]:
# Verify all 48 WC 2026 teams are represented in active_players_df

WC_TEAMS = sorted([
    # Hosts
    "Canada", "Mexico", "United States",
    # UEFA (Europe)
    "England", "France", "Germany", "Spain", "Portugal", "Netherlands",
    "Croatia", "Austria", "Norway", "Scotland", "Switzerland", "Türkiye",
    "Sweden", "Bosnia and Herzegovina", "Czechia",
    # CONMEBOL (South America)
    "Argentina", "Brazil", "Colombia", "Ecuador", "Paraguay", "Uruguay",
    # AFC (Asia)
    "Japan", "Iran", "South Korea", "Qatar", "Saudi Arabia", "Australia",
    "Uzbekistan", "Jordan", "Iraq",
    # CAF (Africa)
    "Morocco", "Senegal", "Algeria", "Egypt", "Côte d'Ivoire", "Nigeria",
    "Tunisia", "Cameroon", "Mali", "DR Congo",
    # CONCACAF (North/Central America & Caribbean)
    "Panama", "Haiti", "Curaçao",
    # OFC (Oceania)
    "New Zealand",
])



In [ ]:
tmkd_df = pd.read_csv("transfermarkt_data/players.csv")
active_players_df = tmkd_df[tmkd_df['last_season'] == 2025]
active_players_df.to_csv("transfermarkt_data/active_players_2025.csv", index=False)

In [17]:
WC_TEAMS = [
    # Hosts
    "Canada", "Mexico", "United States",
    # UEFA (Europe)
    "England", "France", "Germany", "Spain", "Portugal", "Netherlands",
    "Croatia", "Austria", "Norway", "Scotland", "Switzerland", "Belgium",
    "Türkiye", "Sweden", "Bosnia-Herzegovina", "Czech Republic",
    # CONMEBOL (South America)
    "Argentina", "Brazil", "Colombia", "Ecuador", "Paraguay", "Uruguay",
    # AFC (Asia)
    "Japan", "Iran", "Korea, South", "Qatar", "Saudi Arabia", "Australia",
    "Uzbekistan", "Jordan", "Iraq",
    # CAF (Africa)
    "Morocco", "Senegal", "Algeria", "Egypt", "Cote d'Ivoire",
    "Tunisia", "South Africa", "DR Congo", "Cape Verde", "Ghana",
    # CONCACAF (North/Central America & Caribbean)
    "Panama", "Haiti", "Curacao",
    # OFC (Oceania)
    "New Zealand",
]

COUNTRY_ALIASES = {
    # Common alternate spellings that may appear in data sources
    "United States": "United States",
    "USA": "United States",
    "Ivory Coast": "Côte d'Ivoire",
    "Korea Republic": "South Korea",
    "Republic of Korea": "South Korea",
    "Turkey": "Türkiye",
    "Curacao": "Curaçao",
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Czech Republic": "Czechia",
    "Congo DR": "DR Congo",
    "Equador": "Ecuador",
}

nations_in_data = set(active_players_df["country_of_citizenship"].dropna().unique())

present  = sorted([t for t in WC_TEAMS if t in nations_in_data])
missing  = sorted([t for t in WC_TEAMS if t not in nations_in_data])

print(f"Total WC teams:    {len(WC_TEAMS)}")
print(f"Present in data:   {len(present)}")
print(f"Missing from data: {len(missing)}")

if missing:
    print("\n--- MISSING TEAMS ---")
    for t in missing:
        print(f"  {t}")
else:
    print("\nAll 48 teams are present in active_players_df.")

print("\n--- PRESENT TEAMS ---")
for t in present:
    n = active_players_df[active_players_df["country_of_citizenship"] == t].shape[0]
    print(f"  {t:<30}  {n:>5} players")

active_players_df_nation_filtered = active_players_df[active_players_df['country_of_citizenship'].isin(WC_TEAMS)]
active_players_df_nation_filtered.to_csv("transfermarkt_data/active_players_2025_nation_filtered.csv", index=False)
print("Saved: transfermarkt_data/active_players_2025_nation_filtered.csv")

Total WC teams:    48
Present in data:   48
Missing from data: 0

All 48 teams are present in active_players_df.

--- PRESENT TEAMS ---
  Algeria                            98 players
  Argentina                        1002 players
  Australia                         294 players
  Austria                           257 players
  Belgium                           277 players
  Bosnia-Herzegovina                101 players
  Brazil                            955 players
  Canada                             94 players
  Cape Verde                         31 players
  Colombia                          589 players
  Cote d'Ivoire                     134 players
  Croatia                           279 players
  Curacao                            20 players
  Czech Republic                    315 players
  DR Congo                           44 players
  Ecuador                            65 players
  Egypt                              66 players
  England                           284 players


In [18]:
# Optional: merge Understat rows with Transfermarkt (fuzzy name + nation alignment)
# Use `active_players_2025.csv` for best coverage; `*_nation_filtered.csv` may omit
# dual-nation cases (e.g. Matty Cash is Poland on TM but may be missing from WC-only filter).
# For matched rows, `nation` is replaced by TM `country_of_citizenship`; the original
# Understat value is kept in `nation_understat`. Unmatched rows keep Understat `nation`.
#
# pip install rapidfuzz  (see requirements.txt)

import pandas as pd
from pathlib import Path
from transfermarkt_match import merge_understat_with_transfermarkt, unmatched_pairs

TM_PATH = Path("transfermarkt_data/active_players_2025.csv")  # or active_players_2025_nation_filtered.csv
UNDERSTAT_PATH = Path("wc2026_outputs/understat_big5_player_season_2122_2526_full_merged_intl.csv")

u_df = pd.read_csv(UNDERSTAT_PATH)
tm_df = pd.read_csv(TM_PATH)
understat_tm_merged, tm_report = merge_understat_with_transfermarkt(u_df, tm_df)
print(tm_report)
print(understat_tm_merged.filter(regex="^tm_").notna().sum().head())

OUT_TM = Path("wc2026_outputs/understat_big5_player_season_2122_2526_full_merged_intl_tm.csv")
understat_tm_merged.to_csv(OUT_TM, index=False)
print("Wrote", OUT_TM)

unmatched_pairs(understat_tm_merged).to_csv("wc2026_outputs/understat_tm_unmatched_pairs.csv", index=False)

{'rows_total': 13837, 'rows_matched': 12734, 'unique_player_nation_pairs': 5580, 'unique_pairs_matched': 4906, 'overrides_file': 'transfermarkt_data/player_tm_overrides.csv'}
tm_player_id      12734
tm_first_name     11247
tm_last_name      12734
tm_name           12734
tm_last_season    12734
dtype: int64
Wrote wc2026_outputs/understat_big5_player_season_2122_2526_full_merged_intl_tm.csv


In [24]:
understat_tm_merged_specific = understat_tm_merged[understat_tm_merged['tm_international_caps'] > 0]
understat_tm_merged_specific.to_csv("wc2026_outputs/understat_tm_merged_specific.csv", index=False)
print(f"Saved: wc2026_outputs/understat_tm_merged_specific.csv, len rows: {len(understat_tm_merged_specific)}")
understat_tm_merged_specific_nation_filtered = understat_tm_merged_specific[understat_tm_merged_specific['tm_country_of_citizenship'].isin(WC_TEAMS)]
understat_tm_merged_specific_nation_filtered.drop(columns=['nation_understat'], inplace=True)
understat_tm_merged_specific_nation_filtered.to_csv("wc2026_outputs/understat_tm_merged_specific_nation_filtered.csv", index=False)


Saved: wc2026_outputs/understat_tm_merged_specific.csv, len rows: 10417


[04/07/26 15:19:13] WARNING  /var/folders/2s/tlbj4v1n6dxbg5403wzxdv_40000gn/T/ipykernel_42875/21573 ]8;id=276104;file:///opt/homebrew/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/lib/python3.13/warnings.py\warnings.py]8;;\:]8;id=447022;file:///opt/homebrew/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/lib/python3.13/warnings.py#110\110]8;;\
                             82116.py:5: SettingWithCopyWarning:                                                   
                             A value is trying to be set on a copy of a slice from a DataFrame                     
                                                                                                                   
                             See the caveats in the documentation:                                                 
                             https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#                
                             returning-a-view-versus-a-copy                                                        
                               understat_tm_merged_specific_nation_filtered.drop(columns=['nation_u                
                             nderstat'], inplace=True)                                                             
                                                                                                                   

# Data Visualizations

Same charts as `data_visuals.ipynb` (Plotly): nation totals, strip distribution, faceted top-10s, xG vs goals scatter.

**Data:** `understat_tm_merged_specific_nation_filtered` (TM citizenship in `nation`, intl caps > 0, WC nations). If you have not run the cells above, the setup cell loads `wc2026_outputs/understat_tm_merged_specific_nation_filtered.csv`.

Requires: `pip install plotly` (see `requirements.txt`).

In [26]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

BASE_DIR = Path.cwd()
VIZ_CSV = BASE_DIR / "wc2026_outputs/understat_tm_merged_specific_nation_filtered.csv"

try:
    df = understat_tm_merged_specific_nation_filtered.copy()
except NameError:
    df = pd.read_csv(VIZ_CSV)

# `nation` = Transfermarkt citizenship after the merge pipeline
player_df = (
    df.groupby(["player", "nation"], as_index=False)
    .agg(
        total_xg=("xg", "sum"),
        total_np_xg=("np_xg", "sum"),
        total_goals=("goals", "sum"),
        total_assists=("assists", "sum"),
        total_shots=("shots", "sum"),
        total_minutes=("minutes", "sum"),
        seasons=("season", "nunique"),
    )
    .sort_values("total_xg", ascending=False)
)

nation_df = (
    player_df.groupby("nation", as_index=False)
    .agg(
        nation_total_xg=("total_xg", "sum"),
        num_players=("player", "count"),
        top_player_xg=("total_xg", "max"),
    )
    .sort_values("nation_total_xg", ascending=False)
)

print(
    f"Players: {len(player_df):,}  |  Nations: {player_df['nation'].nunique()}  "
    "(intl caps > 0, WC nations, TM citizenship)"
)
player_df.head(10)

Players: 2,792  |  Nations: 47  (intl caps > 0, WC nations, TM citizenship)


,player,nation,total_xg,total_np_xg,total_goals,total_assists,total_shots,total_minutes,seasons
758,Erling Haaland,Norway,128.231064,106.177592,129,30,532,12399,5
959,Harry Kane,England,126.231913,100.433580,139,34,622,13865,5
1492,Kylian Mbappe-Lottin,France,125.869327,100.328809,139,36,689,12998,5
1908,Mohamed Salah,Egypt,104.432408,83.880922,94,59,561,13818,5
1515,Lautaro Martínez,Argentina,93.851186,85.476892,92,19,538,12121,5
1243,Jonathan Christian David,Canada,90.167628,71.164123,79,17,362,12638,5
2078,Ollie Watkins,England,78.364979,74.559130,69,30,409,14240,5
124,Alexander Sørloth,Norway,67.424406,66.681128,69,11,305,9649,5
119,Alexander Isak,Sweden,67.345410,56.724830,63,12,319,9501,5
125,Alexandre Lacazette,France,67.323345,52.878307,65,16,282,9164,4


## Chart 1 – Total xG per Nation (Top 30)
Bar chart of cumulative xG by nation (Transfermarkt citizenship), Big-5 seasons summed.

In [27]:
_k = min(30, len(nation_df))
top30_nations = nation_df.head(_k).sort_values("nation_total_xg")

fig1 = go.Figure(
    go.Bar(
        x=top30_nations["nation_total_xg"],
        y=top30_nations["nation"],
        orientation="h",
        marker=dict(
            color=top30_nations["nation_total_xg"],
            colorscale="Plasma",
            showscale=True,
            colorbar=dict(title="Total xG"),
        ),
        text=top30_nations["nation_total_xg"].round(1),
        textposition="outside",
        customdata=top30_nations[["num_players", "top_player_xg"]].values,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Total xG: %{x:.1f}<br>"
            "Players in dataset: %{customdata[0]}<br>"
            "Best single player xG: %{customdata[1]:.1f}<extra></extra>"
        ),
    )
)

fig1.update_layout(
    title=dict(
        text="Total xG by Nation (TM citizenship) – WC pool, intl caps > 0",
        font_size=18,
    ),
    xaxis_title="Cumulative xG (all seasons)",
    yaxis_title="",
    height=max(400, 24 * _k + 120),
    template="plotly_dark",
    margin=dict(l=180, r=60, t=70, b=50),
)

fig1.show()

## Chart 2 – Player xG distribution (Top 25 Nations)
Strip plot of each player’s total xG, faceted by nation order (same as `data_visuals.ipynb`).

In [28]:
_k25 = min(25, len(nation_df))
top25_nation_list = nation_df.head(_k25)["nation"].tolist()
bubble_df = player_df[player_df["nation"].isin(top25_nation_list)].copy()
nation_order = nation_df[nation_df["nation"].isin(top25_nation_list)]["nation"].tolist()

fig2 = px.strip(
    bubble_df,
    x="nation",
    y="total_xg",
    color="nation",
    hover_name="player",
    hover_data={
        "total_xg": ":.1f",
        "total_goals": True,
        "total_assists": True,
        "total_shots": True,
        "total_minutes": True,
        "seasons": True,
        "nation": False,
    },
    labels={
        "total_xg": "Total xG",
        "nation": "Nation",
        "total_goals": "Goals",
        "total_assists": "Assists",
        "total_shots": "Shots",
        "total_minutes": "Minutes",
        "seasons": "Seasons",
    },
    category_orders={"nation": nation_order},
    title="Player xG distribution – top nations (WC pool, TM citizenship)",
    height=650,
    template="plotly_dark",
)

fig2.update_traces(marker=dict(size=5, opacity=0.75))
fig2.update_layout(
    showlegend=False,
    xaxis=dict(tickangle=-45),
    yaxis_title="Total xG (all seasons combined)",
    margin=dict(b=120),
)

fig2.show()

## Chart 3 – Top 10 players per nation (faceted)
Horizontal bars per nation for the top 20 nations by total xG (fewer if the dataset has fewer nations).

In [29]:
_k20 = min(20, len(nation_df))
top20_nations = nation_df.head(_k20)["nation"].tolist()

top10_per_nation = (
    player_df[player_df["nation"].isin(top20_nations)]
    .groupby("nation", group_keys=False)
    .apply(lambda g: g.nlargest(10, "total_xg"))
    .reset_index(drop=True)
)

fig3 = px.bar(
    top10_per_nation,
    x="total_xg",
    y="player",
    color="total_xg",
    facet_col="nation",
    facet_col_wrap=4,
    orientation="h",
    color_continuous_scale="Viridis",
    labels={"total_xg": "Total xG", "player": ""},
    hover_data={
        "total_goals": True,
        "total_assists": True,
        "total_shots": True,
        "seasons": True,
        "nation": False,
    },
    title="Top 10 players by xG per nation (WC pool, TM citizenship)",
    height=1800,
    template="plotly_dark",
)

fig3.update_layout(
    coloraxis_showscale=False,
    showlegend=False,
    margin=dict(l=130, t=80, b=40),
)
fig3.update_yaxes(matches=None, showticklabels=True, automargin=True)
fig3.update_xaxes(matches=None, showticklabels=True)
fig3.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig3.show()

[04/13/26 15:10:50] WARNING  /var/folders/2s/tlbj4v1n6dxbg5403wzxdv_40000gn/T/ipykernel_42875/35908 ]8;id=782265;file:///opt/homebrew/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/lib/python3.13/warnings.py\warnings.py]8;;\:]8;id=326958;file:///opt/homebrew/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/lib/python3.13/warnings.py#110\110]8;;\
                             48549.py:7: FutureWarning: DataFrameGroupBy.apply operated on the                     
                             grouping columns. This behavior is deprecated, and in a future version                
                             of pandas the grouping columns will be excluded from the operation.                   
                             Either pass `include_groups=False` to exclude the groupings or                        
                             explicitly select the grouping columns after groupby to silence this                  
                             warning.                                                                              
                               .apply(lambda g: g.nlargest(10, "total_xg"))                                        
                                                                                                                   

## Chart 4 – xG vs goals (scatter)
Players in the top 25 nations with total xG > 1; bubble size = total shots. Dashed line = goals = xG.

In [30]:
scatter_df = player_df[
    (player_df["nation"].isin(top25_nation_list)) & (player_df["total_xg"] > 1)
].copy()
scatter_df["xg_overperformance"] = scatter_df["total_goals"] - scatter_df["total_xg"]

if scatter_df.empty:
    print("No players with total_xg > 1 in the selected nations; skip Chart 4.")
else:
    fig4 = px.scatter(
        scatter_df,
        x="total_xg",
        y="total_goals",
        color="nation",
        size="total_shots",
        size_max=22,
        hover_name="player",
        hover_data={
            "total_xg": ":.1f",
            "total_goals": True,
            "total_assists": True,
            "xg_overperformance": ":.1f",
            "seasons": True,
            "nation": True,
            "total_shots": True,
        },
        labels={
            "total_xg": "Total xG",
            "total_goals": "Total Goals",
            "xg_overperformance": "Goals – xG",
            "total_shots": "Total Shots",
            "seasons": "Seasons",
        },
        category_orders={"nation": nation_order},
        title="xG vs goals per player – top nations (bubble size = total shots)",
        height=700,
        template="plotly_dark",
        opacity=0.8,
    )

    max_val = float(scatter_df[["total_xg", "total_goals"]].max().max()) + 5
    fig4.add_shape(
        type="line",
        x0=0,
        y0=0,
        x1=max_val,
        y1=max_val,
        line=dict(color="white", dash="dash", width=1.2),
    )
    fig4.add_annotation(
        x=max_val * 0.9,
        y=max_val * 0.85,
        text="Perfect xG conversion",
        showarrow=False,
        font=dict(color="lightgrey", size=11),
    )

    fig4.update_layout(
        legend=dict(title="Nation", orientation="v", x=1.01),
        margin=dict(r=180),
    )

    fig4.show()

# Country Head-to-Head data

In [ ]:
df = pd.read_csv('transfermarkt_data/games.csv')
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df[df["date"].dt.year >= 2018]
df = df[df['competition_type'] == 'national_team_competition']

In [39]:
# Fixtures where *both* national teams are in the WC 2026 list (head-to-head among WC nations).
# `national_team_games_2018_2026.csv` only contains `national_team_competition` rows (no separate "friendly" flag here).

from pathlib import Path

from scraper import WC_TEAMS, canonical_country

WC_SET = frozenset(WC_TEAMS)

# Transfermarkt spellings not covered by scraper.COUNTRY_ALIASES
EXTRA_TM_TO_WC_CANON = {
    "Czech Republic": "Czechia",
    "Democratic Republic of the Congo": "DR Congo",
    "Turkiye": "Türkiye",
    "Turkey": "Türkiye",
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Curacao": "Curaçao",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Ivory Coast": "Côte d'Ivoire",
    "USA": "United States",
    "Korea Republic": "South Korea",
    "Korea, South": "South Korea",
}


def tm_nation_to_wc(name):
    if pd.isna(name):
        return None
    s = str(name).strip()
    if not s:
        return None
    s = EXTRA_TM_TO_WC_CANON.get(s, s)
    s = canonical_country(s)
    return s if s in WC_SET else None


NT_PATH = Path("transfermarkt_data/national_team_games_2018_2026.csv")
nt_df = pd.read_csv(NT_PATH)
nt_df = nt_df[nt_df["competition_type"] == "national_team_competition"].copy()

nt_df["home_wc"] = nt_df["home_club_name"].map(tm_nation_to_wc)
nt_df["away_wc"] = nt_df["away_club_name"].map(tm_nation_to_wc)

national_team_games_h2h_wc = nt_df[nt_df["home_wc"].notna() & nt_df["away_wc"].notna()].copy()

OUT_H2H = Path("wc2026_outputs/national_team_games_h2h_wc_teams_2018_2026.csv")
national_team_games_h2h_wc.to_csv(OUT_H2H, index=False)

print(
    f"Saved: {OUT_H2H}\n"
    f"  Rows (WC vs WC): {len(national_team_games_h2h_wc):,}  /  {len(nt_df):,} national-team rows in source"
)
national_team_games_h2h_wc[
    ["date", "home_club_name", "away_club_name", "home_wc", "away_wc", "competition_id", "round"]
].head(10)

Saved: wc2026_outputs/national_team_games_h2h_wc_teams_2018_2026.csv
  Rows (WC vs WC): 152  /  365 national-team rows in source


,date,home_club_name,away_club_name,home_wc,away_wc,competition_id,round
1,2018-06-15,Egypt,Uruguay,Egypt,Uruguay,FIWC,Group A
3,2018-06-20,Uruguay,Saudi Arabia,Uruguay,Saudi Arabia,FIWC,Group A
5,2018-06-25,Saudi Arabia,Egypt,Saudi Arabia,Egypt,FIWC,Group A
6,2018-06-15,Morocco,Iran,Morocco,Iran,FIWC,Group B
7,2018-06-15,Portugal,Spain,Portugal,Spain,FIWC,Group B
8,2018-06-20,Portugal,Morocco,Portugal,Morocco,FIWC,Group B
9,2018-06-20,Iran,Spain,Iran,Spain,FIWC,Group B
10,2018-06-25,Iran,Portugal,Iran,Portugal,FIWC,Group B
11,2018-06-25,Spain,Morocco,Spain,Morocco,FIWC,Group B
12,2018-06-16,France,Australia,France,Australia,FIWC,Group C


In [ ]:
df.to_csv("transfermarkt_data/national_team_games_2018_2026.csv", index=False)

365
